# DrugEx ROCS Quick Start — EGCG

The `rocs_rl_tutorial.ipynb` workflow, adapted to a **single reference molecule,
EGCG** (epigallocatechin gallate), with no transfer learning and no scaffold
constraint.

**Differences from the CCR2 tutorial — all of them**

| # | Change | Why |
|---|---|---|
| 1 | EGCG reference SDF instead of `CCR2_reference_ligands.sdf` | the point of the experiment |
| 2 | No `prepare_models.py`; `agent` and `mutate` both from the Papyrus checkpoint | no supervised fine-tuning |
| 3 | `mutate` is a **separate, frozen** instance — not the same object as `agent` | with `mutate=agent`, `epsilon` has no effect at all |
| 4 | `optimization_mode='combo'` removed | not a parameter of `RDKitROCSScorer` on this branch |
| 5 | Score column read from `scorer.getKey()` | one reference gives `RDKit_Supermol_…`, not `…_5refs_…` |
| 6 | Threshold 0.73 instead of 0.871 | 0.871 came from a CCR2 active/decoy ROC and does not transfer |
| 7 | Dropped the CCR2 threshold-validation cell | there are no EGCG actives/decoys |
| 8 | Added a baseline sample for a before/after comparison | `desired_ratio` alone is scored on 10% of the batch and is too noisy to read |

**RL settings are the tutorial's own** (50 epochs × 500). That matters: an
earlier run of this experiment at 20 × 256 produced no measurable shape change,
while the tutorial's own budget does.

Run under PBS, not on the front node: `qsub submit_minimal.sh`

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

from drugex.data.corpus.vocabulary import VocSmiles
from drugex.training.generators import SequenceRNN
from drugex.training.environment import DrugExEnvironment
from drugex.training.explorers import SequenceExplorer
from drugex.training.monitors import FileMonitor
from drugex.training.rewards import ParetoCrowdingDistance
from drugex.training.scorers.conformer_generators import RDKitConformerGenerator
from drugex.training.scorers.modifiers import SmoothClippedScore
from drugex.training.scorers.properties import Property
from drugex.training.scorers.rocs_rdkit import RDKitROCSScorer

import drugex
print("drugex:", Path(drugex.__file__).resolve())   # must be inside this worktree

HERE = Path.cwd().resolve()
MODELS_PR_PATH = Path("/home/fulopj/drx-run/DrugEx/data/models/pretrained/"
                      "smiles-rnn/Papyrus05.5_smiles_rnn_PT")
OUTPUT_DIR = HERE / "minimal_out"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# RL parameters — the tutorial's own demo values.
RL_EPOCHS = 50
RL_EPSILON = 0.2
RL_N_SAMPLES = 500
N_COMPARE = 1000        # molecules sampled before and after, for the comparison

## 1. The reference molecule

One EGCG conformer: ETKDGv3 with a fixed seed, MMFF94, lowest energy kept.

In [ ]:
EGCG_SMILES = "O=C(O[C@@H]1Cc2c(O)cc(O)cc2O[C@@H]1c1cc(O)c(O)c(O)c1)c1cc(O)c(O)c(O)c1"
EGCG_SDF = OUTPUT_DIR / "EGCG_reference.sdf"

mol = Chem.AddHs(Chem.MolFromSmiles(EGCG_SMILES))
params = AllChem.ETKDGv3()
params.randomSeed = 0xE6C6
AllChem.EmbedMultipleConfs(mol, numConfs=200, params=params)
energies = [(e, i) for i, (conv, e) in enumerate(AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=2000))
            if conv == 0]
best_e, best_i = min(energies)
best = Chem.Mol(mol, False, best_i)
best.SetProp("_Name", "EGCG")
with Chem.SDWriter(str(EGCG_SDF)) as w:
    w.write(best)

print(f"reference: {EGCG_SDF}")
print(f"  heavy atoms {best.GetNumHeavyAtoms()} | MMFF {best_e:.2f} kcal/mol "
      f"| conformers kept 1 of {len(energies)}")
Draw.MolsToGridImage([Chem.MolFromSmiles(EGCG_SMILES)], molsPerRow=1,
                     subImgSize=(320, 280), legends=["EGCG (reference)"])

## 2. Scorers and environment

RDKit ROCS (shape + colour) and synthetic accessibility, combined with
`ParetoCrowdingDistance` — the same scheme the tutorial uses.

`ROCS_THRESHOLD = 0.73`. The tutorial's 0.871 came from a CCR2 active/decoy ROC
analysis and carries no meaning for EGCG; 0.73 is the 85th percentile of what
the untrained model produces against this reference. Under Pareto ranking the
threshold only sets the `Desired` flag — it does not enter the reward.

In [ ]:
ROCS_THRESHOLD = 0.73       # raw TanimotoCombo; no modifier on the ROCS scorer
SA_THRESHOLD = 0.27         # modified; equals raw SAscore 4.5

rocs_scorer = RDKitROCSScorer(
    conformer_generator=RDKitConformerGenerator(
        max_conformers=30,
        max_isomers=4,
        max_heavy_atoms=45,
        max_rotatable_bonds=15,
        num_threads=0,          # all cores: conformer generation runs BEFORE the
                                # scorer pool, so the two never compete
        show_progress=False,
    ),
    references=str(EGCG_SDF),
    score_type="TanimotoCombo",
    use_colors=True,
    show_progress=False,
    n_jobs=-1,
)

sa_scorer = Property("SA")
sa_scorer.setModifier(SmoothClippedScore(lower_x=5, upper_x=3))

env = DrugExEnvironment(
    scorers=[rocs_scorer, sa_scorer],
    thresholds=[ROCS_THRESHOLD, SA_THRESHOLD],
    reward_scheme=ParetoCrowdingDistance(),
)

ROCS_KEY = rocs_scorer.getKey()[0]      # single reference -> RDKit_Supermol_...
print("scorer keys:", env.getScorerKeys())

# Sanity check: EGCG against itself, and a dissimilar control.
chk = env.getScores([EGCG_SMILES, "c1ccccc1"])
print(f"  EGCG self-score {chk[ROCS_KEY].iloc[0]:.3f}   benzene {chk[ROCS_KEY].iloc[1]:.3f}")
print("  (self-score ~1.26, not ~2.0: query conformers are generated "
      "independently of the reference pose)")

## 3. Models

Agent and exploration prior both start from the Papyrus checkpoint — no
fine-tuning. They must be **two separate objects**: `SequenceRNN.evolve`
replaces token probabilities with the `mutate` network's for an `epsilon`
fraction of rows, so if `mutate` *is* the agent there is nothing to explore
with and `epsilon` does nothing.

In [ ]:
voc = VocSmiles.fromFile(str(MODELS_PR_PATH / "Papyrus05.5_smiles_rnn_PT.vocab"),
                         encode_frags=False)

agent = SequenceRNN(voc, is_lstm=True)
agent.loadStatesFromFile(str(MODELS_PR_PATH / "Papyrus05.5_smiles_rnn_PT.pkg"))

prior = SequenceRNN(voc, is_lstm=True)                   # SEPARATE instance
prior.loadStatesFromFile(str(MODELS_PR_PATH / "Papyrus05.5_smiles_rnn_PT.pkg"))
for p in prior.parameters():
    p.requires_grad_(False)
prior.eval()
assert prior is not agent, "mutate must not be the agent, or epsilon is inert"

print(f"vocabulary {voc.size} tokens | device {agent.device}")

## 4. Before: sample from the untrained model

The comparator. Raw samples — invalid and duplicate molecules kept — so that
validity and uniqueness mean what they say.

In [ ]:
def sample_and_score(model, n, label):
    t0 = time.time()
    smiles = model.generate(num_samples=n, batch_size=64, drop_invalid=False,
                            drop_duplicates=False, evaluator=None,
                            progress=False)["SMILES"].tolist()
    sc = env.getScores(smiles)              # Valid + Desired + both objectives
    df = pd.DataFrame({"SMILES": smiles})
    for col in sc.columns:
        df[col] = sc[col].values
    df["canonical"] = [Chem.MolToSmiles(m) if m else None
                       for m in (Chem.MolFromSmiles(s) for s in smiles)]
    print(f"  {label}: {n} raw samples in {(time.time()-t0)/60:.1f} min")
    return df

def summarise(df, label):
    v = df[df.Valid == 1]
    return pd.Series({
        "validity": len(v) / len(df),
        "uniqueness": v.canonical.nunique() / len(df),
        "desired_ratio": df.Desired.mean(),
        "median_combo": v[ROCS_KEY].median(),
        "p90_combo": v[ROCS_KEY].quantile(0.90),
    }, name=label)

df_before = sample_and_score(agent, N_COMPARE, "before")
summarise(df_before, "before").to_frame()

## 5. Reinforcement learning

In [ ]:
explorer = SequenceExplorer(
    agent=agent,
    env=env,
    mutate=prior,              # frozen pretrained prior
    epsilon=RL_EPSILON,
    n_samples=RL_N_SAMPLES,
)

output_base = OUTPUT_DIR / "EGCG_rdkit_reinforced"
monitor = FileMonitor(str(output_base), save_smiles=True, reset_directory=True)

print(f"{RL_EPOCHS} epochs, epsilon={RL_EPSILON}, {RL_N_SAMPLES} samples/epoch")
start = time.time()
explorer.fit(monitor=monitor, epochs=RL_EPOCHS)
monitor.close()
print(f"\nCompleted in {(time.time()-start)/60:.1f} minutes")

In [ ]:
df_fit = pd.read_csv(f"{output_base}_fit.tsv", sep="\t")
df_fit.tail(5)

In [ ]:
import matplotlib.pyplot as plt

df_fit.plot(x="Epoch", y=["desired_ratio", "avg_amean"], subplots=True, layout=(1, 2),
            figsize=(12, 4), ylim=(0, 1.1), grid=True, legend=False,
            title=["Desired Molecules Ratio", "Average Score"])
plt.tight_layout()
plt.show()

## 6. After: sample from the trained model

Same scorer, same reference, same settings as the "before" sample.

In [ ]:
df_after = sample_and_score(agent, N_COMPARE, "after")

comparison = pd.concat([summarise(df_before, "before"),
                        summarise(df_after, "after")], axis=1)
comparison["delta"] = comparison["after"] - comparison["before"]
comparison

### Did it work?

The question is whether the **raw TanimotoCombo distribution moved**. A rise in
`desired_ratio` alone is not enough: the threshold is ours to choose, and a
model that emits the same good molecule repeatedly scores well while learning
nothing useful. So uniqueness is reported alongside.

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(5.5, 3.6))
for df, label, color in ((df_before, "before (pretrained)", "#4363d8"),
                         (df_after, "after (reinforced)", "#f58231")):
    sns.kdeplot(df[df.Valid == 1][ROCS_KEY], ax=ax, color=color, fill=True,
                alpha=0.25, linewidth=1.4, cut=0, label=label)
ax.axvline(ROCS_THRESHOLD, color="#b0b6bd", linestyle="--", linewidth=1.0)
ax.set_xlabel("ROCS TanimotoCombo (raw, higher = more EGCG-like)")
ax.set_ylabel("Density")
ax.legend(frameon=False)
sns.despine(ax=ax)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "before_after_tanimotocombo.png", dpi=300)
plt.show()

In [ ]:
# Top molecules from the trained model, beside the reference.
top = (df_after[df_after.Valid == 1]
       .sort_values(ROCS_KEY, ascending=False)
       .drop_duplicates("canonical").head(5))
mols = [Chem.MolFromSmiles(EGCG_SMILES)] + [Chem.MolFromSmiles(s) for s in top.canonical]
legends = ["EGCG (reference)"] + [f"combo={r[ROCS_KEY]:.3f}" for _, r in top.iterrows()]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(260, 240), legends=legends)

In [ ]:
df_before.to_csv(OUTPUT_DIR / "before_raw.csv.gz", index=False)
df_after.to_csv(OUTPUT_DIR / "after_raw.csv.gz", index=False)
comparison.to_csv(OUTPUT_DIR / "comparison.csv")
print(f"written to {OUTPUT_DIR}")